# 14 — Collections Framework

## Objectives
- Master Java's Collections hierarchy
- Choose the right collection for each use case
- Use Collections utility methods
- Apply streams with collections

## Collections Hierarchy
```
Collection
├── List (ordered, duplicates allowed)
│   ├── ArrayList   ← most used
│   └── LinkedList  ← deque/queue
├── Set (unique elements)
│   ├── HashSet     ← O(1), unordered
│   ├── LinkedHashSet ← insertion order
│   └── TreeSet     ← sorted
└── Queue
    ├── PriorityQueue ← heap
    └── ArrayDeque  ← stack/deque

Map (key-value pairs)
├── HashMap     ← O(1), unordered
├── LinkedHashMap ← insertion order
└── TreeMap     ← sorted by key
```

In [1]:
import java.util.*;
import java.util.stream.*;

// === ArrayList ===
List<String> cities = new ArrayList<>(Arrays.asList("Mumbai","Delhi","Bangalore","Chennai","Hyderabad"));
cities.add("Pune");
cities.sort(Comparator.naturalOrder());
System.out.println("Sorted cities: " + cities);
System.out.println("Contains Pune: " + cities.contains("Pune"));

// === HashMap ===
Map<String, Integer> scores = new HashMap<>();
scores.put("Alice", 95); scores.put("Bob", 78); scores.put("Charlie", 88);
scores.merge("Alice", 5, Integer::sum); // Alice +5 bonus
System.out.println("\nScores: " + scores);

String top = scores.entrySet().stream()
    .max(Map.Entry.comparingByValue())
    .map(Map.Entry::getKey).orElse("None");
System.out.println("Top scorer: " + top + " (" + scores.get(top) + ")");

// === PriorityQueue ===
PriorityQueue<Integer> pq = new PriorityQueue<>(Collections.reverseOrder()); // max heap
pq.addAll(Arrays.asList(30, 10, 50, 20, 40));
System.out.println("\nProcessing by priority (descending):");
while (!pq.isEmpty()) System.out.print(pq.poll() + " ");
System.out.println();

// === Set operations ===
Set<String> java = new HashSet<>(Arrays.asList("OOP","Generics","Streams","Collections","JDBC"));
Set<String> python = new HashSet<>(Arrays.asList("OOP","List Comp","Generators","Collections","Numpy"));

Set<String> common = new HashSet<>(java);
common.retainAll(python);
System.out.println("\nCommon topics: " + common);

// === Streams with Collections ===
List<Integer> nums = Arrays.asList(1,2,3,4,5,6,7,8,9,10);
int sumOfEvens = nums.stream().filter(n -> n % 2 == 0).mapToInt(Integer::intValue).sum();
System.out.println("Sum of evens 1-10: " + sumOfEvens);

Map<Boolean, List<Integer>> partitioned = nums.stream()
    .collect(Collectors.partitioningBy(n -> n % 2 == 0));
System.out.println("Even: " + partitioned.get(true));
System.out.println("Odd : " + partitioned.get(false));

Sorted cities: [Bangalore, Chennai, Delhi, Hyderabad, Mumbai, Pune]
Contains Pune: true

Scores: {Bob=78, Alice=100, Charlie=88}
Top scorer: Alice (100)

Processing by priority (descending):
50 40 30 20 10 

Common topics: [OOP, Collections]
Sum of evens 1-10: 30
Even: [2, 4, 6, 8, 10]
Odd : [1, 3, 5, 7, 9]


## Mini Challenge
Given a list of `Employee` records, group them by department using `Collectors.groupingBy()` and find the highest salary in each department.

In [2]:
public record Employee(String name, String department, double salary) {}

In [3]:
import java.util.*;
import java.util.stream.*;

// 1. Create dummy data
List<Employee> employees = Arrays.asList(
    new Employee("Alice", "Engineering", 110000),
    new Employee("Bob", "Engineering", 125000),
    new Employee("Charlie", "HR", 70000),
    new Employee("David", "HR", 85000),
    new Employee("Eve", "Marketing", 90000),
    new Employee("Frank", "Engineering", 105000)
);

// 2. Stream, group, and find max salary
Map<String, Optional<Employee>> maxSalaryByDept = employees.stream()
    .collect(Collectors.groupingBy(
        Employee::department,
        Collectors.maxBy(Comparator.comparingDouble(Employee::salary))
    ));

// 3. Print results nicely
System.out.println("### Highest Salary Per Department ###\n");
maxSalaryByDept.forEach((dept, empOpt) -> {
    empOpt.ifPresent(emp -> 
        System.out.printf("Department: %-15s -> Top Earner: %s ($%,.2f)%n", 
            dept, emp.name(), emp.salary())
    );
});

### Highest Salary Per Department ###

Department: Engineering     -> Top Earner: Bob ($125,000.00)
Department: HR              -> Top Earner: David ($85,000.00)
Department: Marketing       -> Top Earner: Eve ($90,000.00)


In [4]:
Map<String, Double> maxSalaryFiguresOnly = employees.stream()
    .collect(Collectors.groupingBy(
        Employee::department,
        Collectors.collectingAndThen(
            Collectors.maxBy(Comparator.comparingDouble(Employee::salary)),
            empOpt -> empOpt.map(Employee::salary).orElse(0.0)
        )
    ));

System.out.println("\nPure Salary Mapping:\n" + maxSalaryFiguresOnly);



Pure Salary Mapping:
{Engineering=125000.0, HR=85000.0, Marketing=90000.0}
